# Particle Collisions and Emergent Boltzmann-Like Mixing

A gas can be modeled as particles moving freely between elastic collisions.
Even simple local collision rules produce global mixing and statistical regularization.

We simulate 2D hard-disc particles with reflective boundaries and pairwise elastic swaps along collision frames.


## Environment

Particles are represented in complex form $x+iy$ for compact vectorized kinematics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

plt.rcParams["figure.dpi"] = 120
rng = np.random.default_rng(3)


## Initialization

Sample positions and small random velocities.


In [ ]:
k = 90
radius = 0.028
x = rng.uniform(radius, 1 - radius, k) + 1j * rng.uniform(radius, 1 - radius, k)
v = 0.006 * (rng.standard_normal(k) + 1j * rng.standard_normal(k))
v /= np.abs(v) + 1e-12

colors = plt.cm.tab20(np.linspace(0, 1, k))


## Elastic collision step

Exchange velocity components in collision frame (normal/tangent decomposition).


In [ ]:
def dotc(a, b):
    return np.real(a * np.conj(b))

def step_particles(x, v, radius):
    x = x + v
    # wall reflections
    bad = (np.real(x) < radius) | (np.real(x) > 1 - radius)
    v[bad] = -np.real(v[bad]) + 1j * np.imag(v[bad])
    bad = (np.imag(x) < radius) | (np.imag(x) > 1 - radius)
    v[bad] = np.real(v[bad]) - 1j * np.imag(v[bad])
    x = np.clip(np.real(x), radius, 1 - radius) + 1j * np.clip(np.imag(x), radius, 1 - radius)

    D = np.abs(x[:, None] - x[None, :])
    A, B = np.triu_indices(len(x), k=1)
    pairs = np.where(D[A, B] < 2 * radius)[0]
    for idx in pairs:
        a, b = A[idx], B[idx]
        rel = v[a] - v[b]
        if np.abs(rel) < 1e-12:
            continue
        u = rel / np.abs(rel)
        ut = 1j * u
        va = dotc(v[a], ut) * ut + dotc(v[b], u) * u
        vb = dotc(v[b], ut) * ut + dotc(v[a], u) * u
        v[a], v[b] = va, vb
    return x, v


## Simulate and store frames

Keep sparse snapshots to visualize mixing.


In [ ]:
steps = 3800
every = 35
frames = []
xx, vv = x.copy(), v.copy()
for it in range(steps):
    xx, vv = step_particles(xx, vv, radius)
    if it % every == 0:
        frames.append(xx.copy())

print("Stored frames:", len(frames))


## Interactive rendering

Observe diffusion of colored particles over time.


In [ ]:
def show_frame(t=0):
    z = frames[t]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(np.real(z), np.imag(z), s=(radius * 1600), c=colors, edgecolors="none")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")
    ax.set_title(f"Particle system, frame {t}")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(alpha=0.15)
    plt.show()

interact(show_frame, t=IntSlider(min=0, max=len(frames)-1, step=1, value=0));


## Bibliographical resources

- C. Cercignani, *The Boltzmann Equation and Its Applications*.
- L. E. Reichl, *A Modern Course in Statistical Physics*.
- D. Chandler, *Introduction to Modern Statistical Mechanics*.
